# NB05b — PTSAM vs Zero-Shot Eval on Potsdam

**Runs both modes in one pass.** Backbone forward runs once per crop; text/grounding heads run separately for zero-shot and PTSAM.

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `dummyirl/6isprs` — Potsdam GeoTIFF + labels
- `harish77718/ptsam-soft-prompts` — your `soft_prompts.pt`

**Output:** side-by-side mIoU table + 4-col visualization (RGB | GT | zero-shot | PTSAM)

## 1 — Environment setup (~5 min)

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop('PYTHONPATH', None)
os.environ['PATH'] = '/tmp/miniconda/bin:' + os.environ['PATH']

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 tifffile matplotlib -q

## 2 — Clone repo + install model dependencies

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path('/tmp/SegEarth-OV-3')

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    print(f'Updated -> {REPO}')
else:
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/HarishDeepak/rg-segearth-ov3', str(REPO)],
        check=True)
    print(f'Cloned -> {REPO}')

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Eval: zero-shot vs PTSAM on val tile 6_15

- Backbone runs **once per crop** (shared between both modes)
- Score assembly mirrors `segearthov3_segmentor._inference_single_view` exactly:
  `max(instance_logit * object_score, sigmoid(semantic_seg)) * presence_score`
- Per class: max over all synonyms in `cls_potsdam.txt`

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, torch, torch.nn.functional as F
import numpy as np, tifffile, math
from pathlib import Path
from PIL import Image

sys.stdout.reconfigure(line_buffering=True)

DEVICE               = "cuda"
POTSDAM_ROOT         = Path("/kaggle/input/datasets/dummyirl/6isprs")
VAL_TILE             = "6_15"
N_CLASSES            = 6
IGNORE_IDX           = 255
CROP_SIZE            = 1024
STRIDE               = 1024
CONFIDENCE_THRESHOLD = 0.1   # matches cfg_potsdam.py
PROB_THD             = 0.1
BG_IDX               = 5    # clutter

# ── load soft prompts ─────────────────────────────────────────────────────────
candidates = list(Path("/kaggle/input").rglob("soft_prompts.pt"))
if not candidates:
    candidates = [p for p in Path("/kaggle/input").rglob("*.pt") if "sam3" not in p.name]
if not candidates:
    print("WARNING: soft_prompts.pt not found — PTSAM mode will be skipped", flush=True)
    soft_prompts = None
else:
    SOFT_PATH = candidates[0]
    print(f"Found soft_prompts: {SOFT_PATH}", flush=True)
    soft_prompts = torch.load(str(SOFT_PATH), weights_only=True).to(DEVICE)
    print(f"soft_prompts shape: {soft_prompts.shape}", flush=True)

# ── load model ────────────────────────────────────────────────────────────────
from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

processor = Sam3Processor(model, confidence_threshold=CONFIDENCE_THRESHOLD, device=DEVICE)

# ── parse multi-synonym class file ────────────────────────────────────────────
# Official format: each line has comma-separated synonyms for one class.
# Run separate inference per synonym; take max per class.
def get_cls_idx(path):
    words, indices = [], []
    for cls_idx, line in enumerate(Path(path).read_text().splitlines()):
        for syn in line.split(','):
            syn = syn.strip()
            if syn:
                words.append(syn)
                indices.append(cls_idx)
    return words, indices

query_words, query_class_idx = get_cls_idx("./configs/cls_potsdam.txt")
print(f"Queries: {len(query_words)} synonyms -> {N_CLASSES} classes", flush=True)

# ── pre-cache text embeddings for both modes ──────────────────────────────────
# Zero-shot: plain text embeddings
# PTSAM: same text embeddings + soft_prompts appended to language_features
print("Caching text embeddings (zero-shot + PTSAM)...", flush=True)
cached_zs = []   # zero-shot
cached_pt = []   # PTSAM (None if soft_prompts unavailable)

with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    for word in query_words:
        te_zs = model.backbone.forward_text([word], device=DEVICE)
        cached_zs.append({k: v.float().cpu() for k, v in te_zs.items()})

        if soft_prompts is not None:
            te_pt = model.backbone.forward_text([word], device=DEVICE)
            soft_m = torch.ones(1, soft_prompts.shape[0], device=DEVICE,
                                dtype=te_pt["language_mask"].dtype)
            te_pt["language_features"] = torch.cat([te_pt["language_features"], soft_prompts], dim=0)
            te_pt["language_mask"]     = torch.cat([te_pt["language_mask"], soft_m], dim=1)
            cached_pt.append({k: v.float().cpu() for k, v in te_pt.items()})

if not cached_pt:
    cached_pt = None

# ── label helpers ─────────────────────────────────────────────────────────────
RGB_TO_IDX = {
    (255,255,255): 0,  # impervious
    (  0,  0,255): 1,  # building
    (  0,255,255): 2,  # low veg
    (  0,255,  0): 3,  # tree
    (255,255,  0): 4,  # car
    (255,  0,  0): 5,  # clutter
}
def rgb_to_gt(path):
    arr = tifffile.imread(str(path))
    if arr.ndim == 2:
        gt = torch.from_numpy(arr.astype(np.int64))
        if gt.max() > 5:  # 1-indexed
            gt = gt - 1
            gt[gt < 0] = IGNORE_IDX
        return gt
    h, w = arr.shape[:2]
    gt = torch.full((h, w), IGNORE_IDX, dtype=torch.int64)
    for (r, g, b), idx in RGB_TO_IDX.items():
        m = (arr[:,:,0]==r) & (arr[:,:,1]==g) & (arr[:,:,2]==b)
        gt[m] = idx
    return gt

# ── load tile ─────────────────────────────────────────────────────────────────
img_path = next(POTSDAM_ROOT.rglob(f"*{VAL_TILE}*_RGB.tif"), None)
lbl_path = next(POTSDAM_ROOT.rglob(f"*{VAL_TILE}*_label_noBoundary.tif"), None)
if img_path is None:
    print("ERROR: val tile not found. Available tifs:")
    for p in sorted(POTSDAM_ROOT.rglob("*.tif"))[:10]: print(" ", p)
    raise SystemExit(1)

img_arr = tifffile.imread(str(img_path))[:,:,:3]
gt_full  = rgb_to_gt(lbl_path)
H_full, W_full = img_arr.shape[:2]
print(f"Tile: {img_path.name}  {W_full}x{H_full}", flush=True)
print(f"Valid GT pixels: {(gt_full != IGNORE_IDX).sum():,}", flush=True)

# ── score assembly (mirrors segearthov3_segmentor._inference_single_view) ─────
def collect_class_scores(state, h, w, te_cache):
    """Run all synonyms against pre-computed backbone state; return [N_CLASSES,H,W]."""
    logits = torch.zeros((N_CLASSES, h, w), device=DEVICE)
    for te_cpu, cls_idx in zip(te_cache, query_class_idx):
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items():
            state["backbone_out"][k] = v.to(DEVICE)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)

        scores = torch.zeros((h, w), device=DEVICE)

        # Instance-level masks (filtered by confidence threshold)
        if state.get("masks_logits") is not None and state["masks_logits"].shape[0] > 0:
            for inst_id in range(state["masks_logits"].shape[0]):
                inst_logit = state["masks_logits"][inst_id].squeeze()
                inst_score = state["object_score"][inst_id]
                if inst_logit.shape != (h, w):
                    inst_logit = F.interpolate(
                        inst_logit.view(1, 1, *inst_logit.shape),
                        size=(h, w), mode="bilinear", align_corners=False).squeeze()
                scores = torch.max(scores, inst_logit * inst_score)

        # Semantic mask — already sigmoid'd by _forward_grounding
        sem = state["semantic_mask_logits"].squeeze()
        if sem.shape != (h, w):
            sem = F.interpolate(
                sem.view(1, 1, *sem.shape),
                size=(h, w), mode="bilinear", align_corners=False).squeeze()
        scores = torch.max(scores, sem)

        # Presence score
        scores = scores * state["presence_score"]

        # Max across synonyms for same class
        logits[cls_idx] = torch.max(logits[cls_idx], scores)
    return logits

# ── Gaussian kernel ───────────────────────────────────────────────────────────
def make_gaussian_kernel(h, w, dev):
    sy, sx = h / 4.0, w / 4.0
    y = torch.arange(h, device=dev).float() - (h-1) / 2.0
    x = torch.arange(w, device=dev).float() - (w-1) / 2.0
    return torch.exp(-y[:,None]**2 / (2*sy**2)) * torch.exp(-x[None,:]**2 / (2*sx**2))

# ── sliding window — both modes, one backbone pass per crop ───────────────────
h_grids = max(H_full - CROP_SIZE + STRIDE - 1, 0) // STRIDE + 1
w_grids = max(W_full - CROP_SIZE + STRIDE - 1, 0) // STRIDE + 1
total   = h_grids * w_grids
print(f"Sliding window: {h_grids}x{w_grids}={total} crops (size={CROP_SIZE}, stride={STRIDE})", flush=True)

gauss_k    = make_gaussian_kernel(CROP_SIZE, CROP_SIZE, DEVICE)
pred_zs    = torch.zeros(N_CLASSES, H_full, W_full, device=DEVICE)
pred_pt    = torch.zeros(N_CLASSES, H_full, W_full, device=DEVICE) if cached_pt else None
wt_mat     = torch.zeros(H_full, W_full, device=DEVICE)

for hi in range(h_grids):
    for wi in range(w_grids):
        y1 = hi * STRIDE
        x1 = wi * STRIDE
        y2 = min(y1 + CROP_SIZE, H_full)
        x2 = min(x1 + CROP_SIZE, W_full)
        y1 = max(y2 - CROP_SIZE, 0)
        x1 = max(x2 - CROP_SIZE, 0)

        crop_pil = Image.fromarray(img_arr[y1:y2, x1:x2])
        h_c, w_c = y2 - y1, x2 - x1

        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            # Backbone forward — shared by both modes
            state = processor.set_image(crop_pil)

            # Zero-shot
            zs_logits = collect_class_scores(state, h_c, w_c, cached_zs).float()

            # PTSAM
            if cached_pt is not None:
                pt_logits = collect_class_scores(state, h_c, w_c, cached_pt).float()

        g = gauss_k[:h_c, :w_c]
        pred_zs[:, y1:y2, x1:x2] += zs_logits * g.unsqueeze(0)
        if cached_pt is not None:
            pred_pt[:, y1:y2, x1:x2] += pt_logits * g.unsqueeze(0)
        wt_mat[y1:y2, x1:x2] += g

        done = hi * w_grids + wi + 1
        if done % 5 == 0 or done == total:
            print(f"  {done}/{total} crops", flush=True)

def finalize(pred_raw, wt):
    p = pred_raw / wt.unsqueeze(0)
    seg = p.argmax(0)
    seg[p.max(0)[0] < PROB_THD] = BG_IDX
    return seg.cpu()

seg_zs = finalize(pred_zs, wt_mat)
seg_pt = finalize(pred_pt, wt_mat) if pred_pt is not None else None

# ── mIoU for both ─────────────────────────────────────────────────────────────
def compute_miou(seg_pred, gt, n_cls=N_CLASSES, ignore=IGNORE_IDX):
    tp = torch.zeros(n_cls); fp = torch.zeros(n_cls); fn = torch.zeros(n_cls)
    valid = (gt != ignore)
    for c in range(n_cls):
        pc = (seg_pred == c) & valid
        lc = (gt       == c) & valid
        tp[c] = (pc & lc).sum().float()
        fp[c] = (pc & ~lc).sum().float()
        fn[c] = (~pc & lc).sum().float()
    iou = tp / (tp + fp + fn + 1e-6)
    acc = tp / (tp + fn + 1e-6)
    return iou, acc

iou_zs, acc_zs = compute_miou(seg_zs, gt_full)
iou_pt, acc_pt = (compute_miou(seg_pt, gt_full) if seg_pt is not None else (None, None))

CLS_SHORT = ["impervious", "building", "low_veg", "tree", "car", "clutter"]

print(f"\n=== Tile {VAL_TILE} — comparison ===")
if iou_pt is not None:
    print(f"  {'class':<20} {'ZS-IoU':>8}  {'PT-IoU':>8}  {'delta':>7}")
    print(f"  {'-'*52}")
    for n, iz, ip in zip(CLS_SHORT, iou_zs, iou_pt):
        print(f"  {n:<20} {iz*100:7.2f}%  {ip*100:7.2f}%  {(ip-iz)*100:+6.2f}%")
    print(f"  {'-'*52}")
    miou_zs = iou_zs.mean().item()*100
    miou_pt = iou_pt.mean().item()*100
    print(f"  {'mIoU':<20} {miou_zs:7.2f}%  {miou_pt:7.2f}%  {miou_pt-miou_zs:+6.2f}%")
    print(f"  {'mAcc':<20} {acc_zs.mean()*100:7.2f}%  {acc_pt.mean()*100:7.2f}%")
    print(f"\nLiterature zero-shot: 57.8%")
    print(f"Our zero-shot: {miou_zs:.2f}%  ({miou_zs-57.8:+.2f}% vs lit.)")
    print(f"PTSAM:         {miou_pt:.2f}%  ({miou_pt-57.8:+.2f}% vs lit., {miou_pt-miou_zs:+.2f}% vs our ZS)")
else:
    miou_zs = iou_zs.mean().item()*100
    print(f"  {'class':<20} {'IoU':>8}  {'Recall':>8}")
    print(f"  {'-'*40}")
    for n, iz, az in zip(CLS_SHORT, iou_zs, acc_zs):
        print(f"  {n:<20} {iz*100:7.2f}%  {az*100:7.2f}%")
    print(f"  {'-'*40}")
    print(f"  {'mIoU':<20} {miou_zs:7.2f}%")
    print(f"\nLiterature zero-shot: 57.8%  delta: {miou_zs-57.8:+.2f}%")

# ── visualization ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PALETTE = np.array([[255,255,255],[0,0,255],[0,255,255],[0,255,0],[255,255,0],[255,0,0]], dtype=np.uint8)

def to_rgb(label_np):
    out = np.zeros((*label_np.shape, 3), dtype=np.uint8)
    for c in range(N_CLASSES):
        out[label_np == c] = PALETTE[c]
    out[label_np == IGNORE_IDX] = [80, 80, 80]
    return out

from PIL import Image as PILImage
SCALE = max(H_full, W_full) / 1500
def thumb(arr):
    h, w = arr.shape[:2]
    return np.array(PILImage.fromarray(arr).resize((int(w/SCALE), int(h/SCALE)), PILImage.NEAREST))

gt_rgb   = to_rgb(gt_full.numpy())
zs_rgb   = to_rgb(seg_zs.numpy())
pt_rgb   = to_rgb(seg_pt.numpy()) if seg_pt is not None else zs_rgb

N_COLS = 4  # RGB | GT | ZS | PTSAM
CROPS = [
    ("top-left",     0,             0),
    ("centre",       H_full//2-512, W_full//2-512),
    ("bottom-right", H_full-1024,   W_full-1024),
]
legend_patches = [mpatches.Patch(color=PALETTE[i]/255., label=CLS_SHORT[i]) for i in range(N_CLASSES)]

miou_pt_val = iou_pt.mean().item()*100 if iou_pt is not None else float('nan')
col_titles  = [
    "RGB",
    "Ground Truth",
    f"Zero-shot ({miou_zs:.1f}%)",
    f"PTSAM ({miou_pt_val:.1f}%)" if seg_pt is not None else "PTSAM (n/a)",
]

fig, axes = plt.subplots(4, N_COLS, figsize=(N_COLS*6, 24),
                         gridspec_kw={"hspace": 0.05, "wspace": 0.02})

for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=13, fontweight="bold", pad=8)

row0_imgs = [thumb(img_arr), thumb(gt_rgb), thumb(zs_rgb), thumb(pt_rgb)]
for j, im in enumerate(row0_imgs):
    axes[0, j].imshow(im)
    axes[0, j].axis("off")
axes[0, 0].set_ylabel("full tile", fontsize=11)

for row, (lbl, y1c, x1c) in enumerate(CROPS, start=1):
    y1c = max(0, min(y1c, H_full - 1024))
    x1c = max(0, min(x1c, W_full - 1024))
    y2c, x2c = y1c + 1024, x1c + 1024
    crop_imgs = [
        img_arr[y1c:y2c, x1c:x2c],
        gt_rgb[y1c:y2c, x1c:x2c],
        zs_rgb[y1c:y2c, x1c:x2c],
        pt_rgb[y1c:y2c, x1c:x2c],
    ]
    for j, im in enumerate(crop_imgs):
        axes[row, j].imshow(im)
        axes[row, j].axis("off")
    axes[row, 0].set_ylabel(lbl, fontsize=11)

fig.legend(handles=legend_patches, loc="lower center", ncol=N_CLASSES,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, 0.01))
fig.suptitle(
    f"Tile {VAL_TILE} — Zero-shot {miou_zs:.2f}%  |  PTSAM {miou_pt_val:.2f}%  (lit. 57.8%)",
    fontsize=14, fontweight="bold", y=1.001)

out_png = Path("/kaggle/working/eval_comparison.png")
fig.savefig(str(out_png), dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nSaved -> {out_png}", flush=True)
PYEOF
